# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from  tqdm.notebook import tqdm
from itertools import product
import numpy as np
from sklearn.metrics import accuracy_score

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df=pd.read_csv("../data/day-of-week-not-scaled.csv")
previous_df=pd.read_csv("../data/dayofweek.csv")
df['dayofweek']=previous_df['dayofweek']

X=df.drop('dayofweek', axis=1)
y=df['dayofweek']

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
svm=SVC(random_state=21, probability=True)
params={
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

In [ ]:
grid=GridSearchCV(svm, param_grid=params, scoring='accuracy', n_jobs=-1)

In [ ]:
grid.fit(X_train, y_train)

GridSearchCV(estimator=SVC(probability=True, random_state=21), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring='accuracy')

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
print("Лучшие параметры:", grid.best_params_)
print("Лучшая accuracy:", grid.best_score_)

results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score', ascending=True)
results[['params', 'mean_test_score']].head(10)

Лучшие параметры: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Лучшая accuracy: 0.8761090458488228


,params,mean_test_score
70,"{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}",0.876109
64,"{'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'rbf'}",0.863500
58,"{'C': 5, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}",0.816018
52,"{'C': 5, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'rbf'}",0.808608
63,"{'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'linear'}",0.721052
60,"{'C': 10, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'linear'}",0.721052
66,"{'C': 10, 'class_weight': None, 'gamma': 'scale', 'kernel': 'linear'}",0.719587
69,"{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'linear'}",0.719587
51,"{'C': 5, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'linear'}",0.706234
48,"{'C': 5, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'linear'}",0.706234


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
dt=DecisionTreeClassifier(random_state=21)
params_dt={
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

In [ ]:
grid_dt=GridSearchCV(dt, param_grid=params_dt, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...]},
             scoring='accuracy')

In [ ]:
print("Лучшие параметры:", grid_dt.best_params_)
print("Лучшая accuracy:", grid_dt.best_score_)

results = pd.DataFrame(grid_dt.cv_results_).sort_values('rank_test_score', ascending=True)
results[['params', 'mean_test_score']].head(10)

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}
Лучшая accuracy: 0.873864794162192


,params,mean_test_score
69,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}",0.873865
73,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 25}",0.873854
70,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22}",0.872378
97,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 49}",0.872372
71,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 23}",0.872372
75,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 27}",0.872372
76,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 28}",0.872372
77,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 29}",0.872372
78,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 30}",0.872372
79,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 31}",0.872372


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
rf=RandomForestClassifier(random_state=21)
params_rf={
    'n_estimators': [5,10,50,100],
    'max_depth': list(range(1,50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

In [ ]:
grid_rf=GridSearchCV(rf, param_grid=params_rf, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...],
                         'n_estimators': [5, 10, 50, 100]},
             scoring='accuracy')

In [ ]:
print("Лучшие параметры:", grid_rf.best_params_)
print("Лучшая accuracy:", grid_rf.best_score_)

results = pd.DataFrame(grid_rf.cv_results_).sort_values('rank_test_score', ascending=True)
results[['params', 'mean_test_score']].head(10)

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}
Лучшая accuracy: 0.9042929918766351


,params,mean_test_score
95,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}",0.904293
115,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 29, 'n_estimators': 100}",0.904290
698,"{'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators': 50}",0.904290
314,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 30, 'n_estimators': 50}",0.903549
711,"{'class_weight': None, 'criterion': 'gini', 'max_depth': 31, 'n_estimators': 100}",0.903547
99,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 25, 'n_estimators': 100}",0.902809
326,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 33, 'n_estimators': 50}",0.902809
767,"{'class_weight': None, 'criterion': 'gini', 'max_depth': 45, 'n_estimators': 100}",0.902806
779,"{'class_weight': None, 'criterion': 'gini', 'max_depth': 48, 'n_estimators': 100}",0.902806
775,"{'class_weight': None, 'criterion': 'gini', 'max_depth': 47, 'n_estimators': 100}",0.902806


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [3]:
param_grid = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': list(range(1, 50, 5)),  
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

all_params = [dict(zip(param_grid.keys(), v)) for v in product(*param_grid.values())]

results = []

for params in tqdm(all_params, desc="GridSearch Progress"):
    model = RandomForestClassifier(**params, random_state=21)
    
    scores = cross_val_score(
        model, X_train, y_train, 
        cv=5, 
        n_jobs=-1, 
        scoring='accuracy'
    )
    
    results.append({
        **params,
        'mean_accuracy': np.mean(scores),
        'std_accuracy': np.std(scores)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values('mean_accuracy', ascending=False)

print("Лучшие параметры:")
print(results_df.iloc[0])

top_score = results_df['mean_accuracy'].iloc[0]
fifth_score = results_df['mean_accuracy'].iloc[4]
score_diff = top_score - fifth_score

print(f"\nРазница между лучшей и 5-й моделью: {score_diff:.4f}")

if score_diff < 0.02:
    print("Разница незначительная - можно рассмотреть более простую модель")
else:
    print("Разница существенная - лучшая модель значительно лучше")

GridSearch Progress:   0%|          | 0/160 [00:00<?, ?it/s]

Лучшие параметры:
n_estimators          100
max_depth              31
class_weight         None
criterion            gini
mean_accuracy    0.903547
std_accuracy      0.01438
Name: 147, dtype: object

Разница между лучшей и 5-й моделью: 0.0015
Разница незначительная - можно рассмотреть более простую модель


In [4]:
results_df

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
147,100,31,None,gini,0.903547,0.014380
159,100,46,None,gini,0.902806,0.010460
151,100,36,None,gini,0.902806,0.010460
155,100,41,None,gini,0.902806,0.010460
111,50,36,None,gini,0.902063,0.014494
...,...,...,...,...,...,...
42,10,1,None,entropy,0.369404,0.019380
3,5,1,None,gini,0.364219,0.021651
2,5,1,None,entropy,0.353832,0.016467
1,5,1,balanced,gini,0.283390,0.011062


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [ ]:
rf=RandomForestClassifier(random_state=21, class_weight= None, criterion='gini', max_depth= 28, n_estimators= 50 )
rf.fit(X_train, y_train)
pred=rf.predict(X_test)
accuracy_score(pred, y_test)

0.9289940828402367